# Anima 引擎部署（Sprint 10：Worker API 轮询 + LLM 4 槽位容灾）

前置：Cloudflare Worker 已部署并配置 `ENGINE_KEY` secret。
本 notebook 在下方单元格填写 ①你的仓库地址（含改造后代码） ②Worker 地址 ③ENGINE_KEY ④LLM 4 槽位配置（第 1 个为主，出错按序切换；共用同一 model）。

In [ ]:
# ===== 必填配置（已写死；后续有变化请直接改这里） =====
REPO_URL = "https://github.com/Reaky-Dawn/AnimaBot.git"  # 你的引擎代码仓库
WORKER_BASE_URL = "https://anima-web.chenzilong315.workers.dev"  # Worker 域名（animadraw.cloud 生效后改为 https://animadraw.cloud）
ENGINE_KEY = "LFP4SoGh6O1Xb5nxQzT3fItkjegwVmsaicEMWDJHCvpNduKZ"  # 与 Worker `wrangler secret put ENGINE_KEY` 一致
ENGINE_ID = "engine-1"

# ===== LLM 配置（Sprint 10：4 个槽位，第 1 个为主，出错按序切换；共用同一模型） =====
# 每个槽位只需 api_key + base_url；model 全局一个（如 deepseek-v4-flash）
MODEL = "deepseek-v4-flash"
PROVIDERS = [
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
]

import os, json, subprocess, sys, time
os.environ["WORKER_BASE_URL"] = WORKER_BASE_URL
os.environ["ENGINE_KEY"] = ENGINE_KEY
os.environ["ENGINE_ID"] = ENGINE_ID

# 启动前校验（已写死，但保留检查以防意外）
assert WORKER_BASE_URL not in ("", "http://127.0.0.1:8787", "https://anima.example.com"), \
    f"WORKER_BASE_URL 仍为占位符（当前={WORKER_BASE_URL}），请修改"
if len(ENGINE_KEY) < 16:
    print('⚠️ 警告：ENGINE_KEY 长度异常，请检查')
print("配置已设置（WORKER_BASE_URL=", WORKER_BASE_URL, "，ENGINE_KEY 长度:", len(ENGINE_KEY), "，LLM 槽位数:", len(PROVIDERS), "）")

In [ ]:
# ===== 启动提速（Sprint 13）：优先从缓存 dataset 解压 ComfyUI（跳过 clone，省 1-2 分钟） =====
# 首次运行没有缓存 → 走慢路径（git clone），随后由「打包缓存」单元格把环境上传为
# dataset reagino/comfyui-cache；之后每次启动直接解压秒级就绪。
import os, pathlib, tarfile, shutil, subprocess

INPUT_DIR = pathlib.Path('/kaggle/input')
CACHE_DIR = INPUT_DIR / 'comfyui-cache'
WORK = pathlib.Path('/kaggle/working')
COMFY = WORK / 'ComfyUI'
TAR_NAME = 'comfyui.tar'

restored = False
if (CACHE_DIR / TAR_NAME).exists():
    print(f'发现缓存 {CACHE_DIR / TAR_NAME}，解压中（约 20-40s）…')
    try:
        with tarfile.open(CACHE_DIR / TAR_NAME) as tf:
            try:
                tf.extractall(WORK, filter='tar')
            except TypeError:
                tf.extractall(WORK)  # 旧版 Python 无 filter 参数
        restored = COMFY.exists()
        print('缓存解压完成:', restored)
    except Exception as e:
        print('⚠️ 缓存解压失败，回退 git clone:', e)
        restored = False

if not restored:
    if COMFY.exists():
        shutil.rmtree(COMFY)
    print('无缓存，git clone ComfyUI（慢路径，约 1-2 分钟）…')
    r = subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/chinokikiss/ComfyUI.git', str(COMFY)],
                       capture_output=True, text=True)
    print('clone ' + ('OK' if COMFY.exists() else '失败: ' + (r.stderr or '')[-300:]))

# AnimaBot 引擎代码：体积小、更新频繁 → 每次都拉最新（不走缓存）
r = subprocess.run(['bash', '-lc', f'rm -rf {WORK}/AnimaBot && git clone {REPO_URL} {WORK}/AnimaBot'],
                   capture_output=True, text=True)
print('AnimaBot clone:', 'OK' if (WORK / 'AnimaBot').exists() else '失败: ' + (r.stderr or '')[-300:])

In [ ]:
# ===== 模型接入（Sprint 13：软链不复制，4.2GB UNet 秒级接入） =====
# 从 dataset reagino/animamodel 把模型软链到 ComfyUI/models/；/kaggle/input 每次会话都挂载，
# 软链天然有效。缓存解压出的旧软链若失效（dataset 改名）会自动重建。
import os, pathlib, zipfile, shutil

INPUT_DIR = pathlib.Path('/kaggle/input')
if not INPUT_DIR.exists() or not any(INPUT_DIR.iterdir()):
    print('⚠️ 未在 /kaggle/input 找到任何输入！请 Add Input 连接你的 dataset（含模型）。')

targets = {
    'diffusion_models': ['miaomiaoHarem_anima12.safetensors'],
    'text_encoders': ['miaomiaoHarem_anima14_txt.safetensors'],
    'vae': ['qwenImage_qwenImageVAE.safetensors'],
    'upscale_models': ['4x-AnimeSharp.safetensors'],
}

def find_model(name):
    """在整个 /kaggle/input 递归找目标模型文件；也支持 zip 内（解压后再链）。"""
    for f in INPUT_DIR.rglob(name):
        if f.is_file():
            return f
    for z in INPUT_DIR.rglob('*.zip'):
        try:
            with zipfile.ZipFile(z) as zf:
                if name in zf.namelist():
                    tmp = pathlib.Path('/kaggle/working/_models_unzip')
                    zf.extract(name, tmp)
                    return tmp / name
        except Exception:
            continue
    return None

missing = []
for subdir, names in targets.items():
    dest_dir = pathlib.Path('/kaggle/working/ComfyUI/models') / subdir
    dest_dir.mkdir(parents=True, exist_ok=True)
    for name in names:
        dst = dest_dir / name
        if dst.is_symlink() and not dst.exists():
            dst.unlink()  # 失效软链（指向的 dataset 挂载点变了）→ 重建
        elif dst.exists():
            print(f'已存在跳过: {name}')
            continue
        src = find_model(name)
        if src is None:
            missing.append(name)
            continue
        try:
            os.symlink(src.resolve(), dst)
            print(f'已软链: {name} -> ComfyUI/models/{subdir}/')
        except OSError:
            shutil.copy2(src, dst)
            print(f'软链失败已复制: {name} -> ComfyUI/models/{subdir}/')

if missing:
    print('⚠️ 缺失以下模型（引擎启动后可能因模型缺失而失败，但不会杀会话）:', missing)
else:
    print('模型接入完成')

In [ ]:
# ===== 依赖安装（Sprint 13：有缓存 wheel 离线装，秒级；无缓存在线装） =====
import pathlib, os

CACHE_DIR = pathlib.Path('/kaggle/input/comfyui-cache')
COMFY_REQ = '/kaggle/working/ComfyUI/requirements.txt'
ANIMA_REQ = '/kaggle/working/AnimaBot/requirements.txt'
has_wheels = CACHE_DIR.exists() and any(CACHE_DIR.glob('*.whl'))

if has_wheels:
    print('从缓存 wheel 离线安装依赖…')
    # torch/triton 等 Kaggle 镜像已预装（wheel 缓存里刻意不含），--no-index 下按已装满足；
    # 若版本 pin 冲突导致离线失败，|| 回退在线安装兜底。
    os.system(f'pip install --no-index --find-links={CACHE_DIR} -r {COMFY_REQ} -q '
              f'|| pip install -r {COMFY_REQ} -q')
    os.system(f'pip install --no-index --find-links={CACHE_DIR} -r {ANIMA_REQ} -q '
              f'|| pip install -r {ANIMA_REQ} -q')
    os.system(f'pip install --no-index --find-links={CACHE_DIR} sageattention -q '
              f'|| pip install sageattention -q')
else:
    print('无缓存 wheel，在线安装依赖（首次约 2-4 分钟）…')
    os.system(f'pip install -r {COMFY_REQ} -q')
    os.system(f'pip install -r {ANIMA_REQ} -q')
    os.system('pip install sageattention -q')

# oxipng：优先用缓存 deb 离线装
deb = list(CACHE_DIR.glob('oxipng*.deb')) if CACHE_DIR.exists() else []
if deb:
    os.system(f'dpkg -i {deb[0]} > /dev/null 2>&1 || apt-get install -y oxipng > /dev/null 2>&1')
else:
    os.system('apt-get install -y oxipng > /dev/null 2>&1 || echo "oxipng 安装失败（可选，跳过压缩）"')
print('依赖安装阶段完成')

In [ ]:
# ===== 打包缓存 dataset（Sprint 13；仅首次运行执行，之后检测到已存在即跳过） =====
# 产物：私有 dataset reagino/comfyui-cache = ComfyUI 目录 tar + pip wheel 缓存 + oxipng deb。
# 之后每次启动解压 tar + 离线装 wheel，节省 clone+pip 约 3-5 分钟（GPU 会话内的墙钟时间）。
# ComfyUI 需要升级时：到 Kaggle 删除该 dataset（或设 FORCE_CACHE_UPDATE=1）再重跑本单元格即可重建。
import os, pathlib, subprocess, tarfile, json, shutil

FORCE_CACHE_UPDATE = os.environ.get('FORCE_CACHE_UPDATE') == '1'
CACHE_SLUG = 'reagino/comfyui-cache'
WORK = pathlib.Path('/kaggle/working')
STAGE = WORK / '_cache_stage'
TAR_NAME = 'comfyui.tar'

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def _skip_git(ti):
    return None if ('/.git' in '/' + ti.name or '__pycache__' in ti.name) else ti

if not os.environ.get('KAGGLE_KEY'):
    print('⚠️ kaggle CLI 未认证（缺 KAGGLE_KEY），跳过缓存打包。本次仍以慢路径运行，不影响功能。')
else:
    r = sh(f'kaggle datasets status {CACHE_SLUG}')
    cache_exists = (r.returncode == 0)
    print('缓存 dataset 状态:', '已存在' if cache_exists else '不存在')

    if FORCE_CACHE_UPDATE or not cache_exists:
        print('开始打包缓存（首次约 3-6 分钟，含上传）…')
        shutil.rmtree(STAGE, ignore_errors=True)
        STAGE.mkdir(parents=True)
        # 1) ComfyUI 目录 tar（排除 .git/__pycache__；模型软链本身只有几十字节）
        with tarfile.open(STAGE / TAR_NAME, 'w') as tf:
            tf.add(WORK / 'ComfyUI', arcname='ComfyUI', filter=_skip_git)
        print('tar 完成:', (STAGE / TAR_NAME).stat().st_size // (1024 * 1024), 'MB')
        # 2) pip wheel 缓存（torch/triton/nvidia 等超大包排除：Kaggle 镜像已预装）
        sh(f'pip download -r {WORK}/ComfyUI/requirements.txt -d {STAGE} -q')
        sh(f'pip download -r {WORK}/AnimaBot/requirements.txt -d {STAGE} -q')
        sh(f'pip download sageattention --no-deps --no-build-isolation -d {STAGE} -q')
        removed = 0
        for f in STAGE.glob('*'):
            if f.suffix == '.whl' and f.name.lower().startswith(('torch', 'triton', 'nvidia')):
                f.unlink(); removed += 1
        print(f'wheel 缓存完成（排除超大包 {removed} 个）')
        # 3) oxipng deb
        sh('apt-get update -qq')
        sh(f'cd {STAGE} && apt-get download oxipng')
        # 4) 元数据 + 上传（sageattention wheel 保留在根目录，恢复端直接 find-links 整个 dataset）
        meta = {'id': CACHE_SLUG, 'title': 'comfyui-cache', 'licenses': [{'name': 'GPL-3.0'}]}
        (STAGE / 'dataset-metadata.json').write_text(json.dumps(meta), encoding='utf-8')
        r = sh(f'kaggle datasets create -p {STAGE}') if not cache_exists else \
            sh(f'kaggle datasets version -p {STAGE} -m "cache update" -r skip')
        print('缓存上传:', (r.stdout or '')[-300:] or (r.stderr or '')[-300:])
        shutil.rmtree(STAGE, ignore_errors=True)
    else:
        print('缓存 dataset 已存在，跳过打包（重建方法见本单元格注释）')

In [ ]:
import json
cfg = {
  "providers": PROVIDERS,
  "model": MODEL,
  "logging": True,
}
with open("/kaggle/working/AnimaBot/config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
print("config.json 已写入（providers:", len(PROVIDERS), "）")

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Sprint 11.5：不再显式启动 ComfyUI——引擎 core.py 自管理：
#   空闲 IDLE_TIMEOUT_SEC 秒无任务 -> 关闭 ComfyUI 释放 GPU（零消耗）
#   有新任务 -> 冷启动 ComfyUI（加载模型约 30-60s），用户需稍等
# 引擎只启动 core.py + 保活阻塞。
log_dir = Path('/kaggle/working/engine_logs')
log_dir.mkdir(exist_ok=True)

base_env = dict(os.environ)
base_env.update({
    "WORKER_BASE_URL": WORKER_BASE_URL,
    "ENGINE_KEY": ENGINE_KEY,
    "ENGINE_ID": ENGINE_ID,
    "IDLE_TIMEOUT_SEC": "300",  # 空闲 5 分钟关 ComfyUI；可改为 60 更快释放
    "ENGINE_WORKERS": "2",      # Sprint 13：并发 worker 数（双 T4 实例各跑一单）
})

out = open(str(log_dir / "engine.log"), "w")
err = open(str(log_dir / "engine.err"), "w")
p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/AnimaBot/core.py"],
                     cwd="/kaggle/working/AnimaBot", env=base_env, stdout=out, stderr=err)
print("已启动引擎 core.py PID", p.pid)

print("\n引擎已启动（ComfyUI 由引擎按需冷启动，空闲自动休眠释放 GPU）")
print("引擎日志: /kaggle/working/engine_logs/engine.log （tail -u 查看实时）")
print("错误明细: /kaggle/working/engine_logs/errors.log （每次任务失败追加完整步骤日志）")

# ===== 有限保活：有任务就一直跑，空闲则结束（节省 GPU 额度） =====
# Sprint 13：状态查询改用 httpx（原 urllib 的 UA 被 Cloudflare 机器人规则 403，
# 导致保活误判「无任务」而提前退出会话）。
import time, httpx
EMPTY_HOLD_MIN = 2   # 队列清空后再保持 2 分钟（防瞬时新任务）
CHECK_INTERVAL = 15  # 每 15s 检查一次

def _fetch_status():
    try:
        r = httpx.get(f"{WORKER_BASE_URL}/api/engine/status",
                      params={"engine_id": ENGINE_ID},
                      headers={"Authorization": "Bearer " + ENGINE_KEY},
                      timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception:
        return {}

print("进入有限保活模式（有任务持续跑；空闲 ", EMPTY_HOLD_MIN, " 分钟后结束，节省 GPU 额度）")
empty_since = None
try:
    while True:
        st = _fetch_status()
        busy = int(st.get('queued_count', 0)) > 0 or int(st.get('active_count', 0)) > 0
        if busy:
            empty_since = None  # 有任务，重置空闲计时
        else:
            if empty_since is None:
                empty_since = time.time()
            elif (time.time() - empty_since) >= EMPTY_HOLD_MIN * 60:
                print(f"队列清空已超 {EMPTY_HOLD_MIN} 分钟，结束保活（释放 GPU 额度）。")
                break
        time.sleep(CHECK_INTERVAL)
except KeyboardInterrupt:
    print("保活结束")
print("notebook 保活结束，会话将停止。有新任务时由 GitHub Actions 自动重启。")